# Build Synaptome form example

This notebook configures and executes a local Build Synaptome backend build.

It generates and validates a single-neuron SONATA circuit locally. Circuit registration, upload, and provenance registration remain intentionally out of scope.

The form is organized around existing reusable OBI-One blocks:

- `MEModelFromID` selects the post-synaptic ME-model.
- `SynapticModelUnion` defines available synaptic physiology models.
- `AllDistributionsUnion` defines reusable distributions referenced by synaptic models.
- `SynapseGroup` defines incoming synapse groups.
- `MorphologyLocationUnion` defines the placement strategy for each synapse group.

In [1]:
import json
import os
from pathlib import Path

import bluepysnap
from entitysdk import Client, ProjectContext
import obi_one as obi
from obi_auth import get_token

from obi_one.core.info import Info
from obi_one.scientific.blocks.distributions.constant import (
    FloatConstantDistribution,
    IntConstantDistribution,
)
from obi_one.scientific.blocks.morphology_locations.path_distance import (
    PathDistanceMorphologyLocations,
)
from obi_one.scientific.blocks.morphology_locations.random import RandomMorphologyLocations
from obi_one.scientific.blocks.synaptic_models.tsodyks_markram import (
    ExcitatoryTsodyksMarkramSynapticModel,
    InhibitoryTsodyksMarkramSynapticModel,
)
from obi_one.scientific.from_id.memodel_from_id import MEModelFromID
from obi_one.scientific.tasks.build_synaptome import (
    BuildSynaptomeScanConfig,
    SynapseGroup,
    build_synaptome,
)
from obi_one.scientific.unions.unions_distributions import AllDistributionsReference
from obi_one.scientific.unions.unions_synaptic_models import SynapticModelReference

## Connect to EntityCore

The backend needs an authenticated `Client` only to resolve and stage the selected ME-model. This example uses the repository's staging test project constants; replace the project context and ME-model ID when targeting another project. No entities are created by this example.

In [5]:
environment = os.getenv("OBI_ENVIRONMENT", "staging")
project_context = ProjectContext(virtual_lab_id=obi.LAB_ID_STAGING_TEST, project_id=obi.PROJECT_ID_STAGING_TEST)
token = get_token(environment=environment)
db_client = Client(
    environment=environment,
    project_context=project_context,
    token_manager=token,
)

## Select the ME-model

Select the existing ME-model entity that provides the postsynaptic morphology, e-model configuration, mechanisms, and template assets.

In [6]:
entity_ID = "3ba6c735-c321-4890-bb8e-60be11777567"

In [ ]:
initialize = BuildSynaptomeScanConfig.Initialize(
    me_model=MEModelFromID(id_str=entity_ID),
)

## Define reusable distributions

Synaptic models reference these dictionary entries by name.

In [ ]:
distributions = {
    "exc_conductance": FloatConstantDistribution(value=0.4),
    "exc_delay": FloatConstantDistribution(value=1.5),
    "inh_conductance": FloatConstantDistribution(value=0.8),
    "inh_delay": FloatConstantDistribution(value=1.0),
    "rrp_vesicles": IntConstantDistribution(value=5),
}

## Define synaptic models

Excitatory or inhibitory behavior is implied by the selected synaptic model type. Synapse groups do not ask for excitatory/inhibitory separately.

In [ ]:
synaptic_models = {
    "exc_tm": ExcitatoryTsodyksMarkramSynapticModel(
        conductance_distribution=AllDistributionsReference(
            block_dict_name="distributions",
            block_name="exc_conductance",
        ),
        delay_distribution=AllDistributionsReference(
            block_dict_name="distributions",
            block_name="exc_delay",
        ),
        n_rrp_vesicles_distribution=AllDistributionsReference(
            block_dict_name="distributions",
            block_name="rrp_vesicles",
        ),
    ),
    "inh_tm": InhibitoryTsodyksMarkramSynapticModel(
        conductance_distribution=AllDistributionsReference(
            block_dict_name="distributions",
            block_name="inh_conductance",
        ),
        delay_distribution=AllDistributionsReference(
            block_dict_name="distributions",
            block_name="inh_delay",
        ),
        n_rrp_vesicles_distribution=AllDistributionsReference(
            block_dict_name="distributions",
            block_name="rrp_vesicles",
        ),
    ),
}

## Define incoming synapse groups

Each group references one synaptic model and owns one morphology-location placement strategy.

In [ ]:
synapse_groups = {
    "basal_exc_inputs": SynapseGroup(
        group_name="Basal excitatory inputs",
        synaptic_model=SynapticModelReference(
            block_dict_name="synaptic_models",
            block_name="exc_tm",
        ),
        placement_strategy=RandomMorphologyLocations(
            number_of_locations=100,
            section_types=(3,),
            random_seed=42,
        ),
    ),
    "apical_inh_inputs": SynapseGroup(
        group_name="Apical inhibitory inputs",
        synaptic_model=SynapticModelReference(
            block_dict_name="synaptic_models",
            block_name="inh_tm",
        ),
        placement_strategy=PathDistanceMorphologyLocations(
            number_of_locations=25,
            section_types=(4,),
            path_dist_mean=250.0,
            path_dist_tolerance=50.0,
            random_seed=7,
        ),
    ),
}

## Build the form config

The UI-facing object remains a scan config. The local backend consumes its resolved single-coordinate form.

In [ ]:
build_synaptome_config = BuildSynaptomeScanConfig(
    info=Info(
        campaign_name="Example Build Synaptome",
        campaign_description="Build a simulatable single-cell SONATA synaptome.",
    ),
    initialize=initialize,
    distributions=distributions,
    synaptic_models=synaptic_models,
    synapse_groups=synapse_groups,
)

## Inspect the JSON payload

This is the payload shape expected by the generated schema/UI.

In [ ]:
payload = build_synaptome_config.model_dump(mode="json")
print(json.dumps(payload, indent=2)[:4000])

## Inspect form sections

The generated schema exposes the same sections used by the beta form renderer.

In [ ]:
schema = BuildSynaptomeScanConfig.model_json_schema()
print(schema["title"])
print(schema.get("group_order"))
print(list(schema["properties"].keys()))

## Generate the SONATA artifact

`build_synaptome` stages the selected ME-model, generates locations and physiology independently for each afferent group, writes virtual nodes and chemical edges, and validates the result through the canonical circuit loader. The output directory must not already exist.

In [ ]:
single_config = build_synaptome_config.cast_to_single_coord()
output_directory = Path("obi-output/build_synaptome")

result = build_synaptome(
    single_config,
    output_directory,
    db_client=db_client,
)
print(result.circuit_config_path)
print(f"Generated {len(result.generated_files)} files")

## Inspect the generated circuit

Loading with BluePySnap validates more than file existence: node and edge populations, references, morphology-placement columns, and physiology columns are all readable.

In [ ]:
circuit = bluepysnap.Circuit(result.circuit_config_path)

print("Node populations:")
for population_name in circuit.nodes.population_names:
    population = circuit.nodes[population_name]
    print(f"  {population_name}: type={population.type}, nodes={population.size}")

print("Edge populations:")
for population_name in circuit.edges.population_names:
    population = circuit.edges[population_name]
    print(f"  {population_name}: edges={population.size}")
    print(f"    properties={sorted(population.property_names)}")
    print(
        population.get(
            population.ids()[:3],
            properties=[
                "@source_node",
                "@target_node",
                "afferent_section_id",
                "afferent_segment_id",
                "afferent_segment_offset",
                "conductance",
                "delay",
                "syn_type_id",
            ],
        )
    )